# Task 1: Titanic Data Cleaning & Preprocessing

**Goal:** Take the raw Titanic passenger data and turn it into a clean dataset that is ready for analysis.

**What we will do (the 4 required tasks):**
1. Handle missing values
2. Remove duplicates
3. Convert data types
4. Rename columns

## Step 0: Load and look at the raw data
Before changing anything, we always look at the data first. We want to know how big it is, what the columns are, and what is wrong with it.

In [1]:
import pandas as pd

df = pd.read_csv("../data/titanic_raw.csv")
print("Rows, columns:", df.shape)
df.head()

Rows, columns: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [2]:
# Column types and how many values are filled in
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [3]:
# How many values are missing in each column?
missing_before = df.isnull().sum()
print(missing_before)
print("Total missing values:", missing_before.sum())

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64
Total missing values: 866


**What we found:**
- `Age` is missing 177 values
- `Cabin` is missing 687 values (about 77% of the column)
- `Embarked` is missing only 2 values

## Step 1: Handle missing values
Each column gets a different fix, depending on how much is missing.

**Age (177 missing):** fill with the **median** age. The median is the middle value, so a few very old passengers do not pull it up the way they would pull up the average. We use the median of each group (same class and same sex), because a 1st-class woman and a 3rd-class man had different typical ages.

In [4]:
df["Age"] = df["Age"].fillna(
    df.groupby(["Pclass", "Sex"])["Age"].transform("median")
)
print("Age missing now:", df["Age"].isnull().sum())

Age missing now: 0


**Embarked (2 missing):** fill with the **mode**, the most common value. With only 2 gaps, this is safe.

In [5]:
print(df["Embarked"].value_counts())
most_common_port = df["Embarked"].mode()[0]
print("Most common port:", most_common_port)

df["Embarked"] = df["Embarked"].fillna(most_common_port)
print("Embarked missing now:", df["Embarked"].isnull().sum())

Embarked
S    644
C    168
Q     77
Name: count, dtype: int64
Most common port: S
Embarked missing now: 0


**Cabin (687 missing):** more than 3 out of 4 values are empty. Filling that many would just be guessing, so we **drop the column**. But whether a passenger *had* a recorded cabin can be a useful clue (it is linked to wealth), so first we keep that as a new yes/no column called `Has_Cabin`.

In [6]:
df["Has_Cabin"] = df["Cabin"].notna().astype(int)   # 1 = has a cabin, 0 = no cabin recorded
df = df.drop(columns=["Cabin"])

print(df.isnull().sum())
print("Total missing values:", df.isnull().sum().sum())

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
Has_Cabin      0
dtype: int64
Total missing values: 0


## Step 2: Remove duplicates
A duplicate is a row that appears more than once. We check for fully identical rows, and also check that every `PassengerId` is unique.

In [7]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate PassengerIds:", df["PassengerId"].duplicated().sum())

rows_before = len(df)
df = df.drop_duplicates()
print("Rows before:", rows_before, "| Rows after:", len(df))

Duplicate rows: 0
Duplicate PassengerIds: 0
Rows before: 891 | Rows after: 891


**Result:** this dataset has no duplicates, so nothing was removed. Checking is still an important part of cleaning.

## Step 3: Convert data types
Some columns are stored as the wrong type. We fix them:
- `Age` is stored with decimals (e.g. 28.0), but ages are whole years, so we convert it to a whole number.
- `Survived`, `Pclass`, `Sex`, `Embarked` are really **categories** (a fixed set of options), not numbers to do maths on.

In [8]:
print("Types BEFORE:")
print(df.dtypes)

Types BEFORE:
PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Embarked           str
Has_Cabin        int64
dtype: object


In [9]:
df["Age"] = df["Age"].round().astype(int)

for col in ["Survived", "Pclass", "Sex", "Embarked"]:
    df[col] = df[col].astype("category")

print("Types AFTER:")
print(df.dtypes)

Types AFTER:
PassengerId       int64
Survived       category
Pclass         category
Name                str
Sex            category
Age               int64
SibSp             int64
Parch             int64
Ticket              str
Fare            float64
Embarked       category
Has_Cabin         int64
dtype: object


**Note:** babies under 1 year old have ages like 0.42. Rounding to whole years makes them 0, which is correct for "age in years".

## Step 4: Rename columns
We give every column a clear name: all lowercase, words joined with underscores, and the confusing ones made readable (`SibSp` and `Parch` are hard to understand).

In [10]:
df = df.rename(columns={
    "PassengerId": "passenger_id",
    "Survived":    "survived",
    "Pclass":      "passenger_class",
    "Name":        "name",
    "Sex":         "sex",
    "Age":         "age",
    "SibSp":       "siblings_spouses",
    "Parch":       "parents_children",
    "Ticket":      "ticket",
    "Fare":        "fare",
    "Embarked":    "embarked_port",
    "Has_Cabin":   "has_cabin",
})
print(list(df.columns))

['passenger_id', 'survived', 'passenger_class', 'name', 'sex', 'age', 'siblings_spouses', 'parents_children', 'ticket', 'fare', 'embarked_port', 'has_cabin']


## Step 5: Make the values readable
Codes like `0/1` and `C/Q/S` are hard to read. We replace them with words. We also remove any accidental extra spaces in the text columns.

In [11]:
df["survived"] = df["survived"].map({0: "No", 1: "Yes"}).astype("category")
df["embarked_port"] = df["embarked_port"].map(
    {"C": "Cherbourg", "Q": "Queenstown", "S": "Southampton"}
).astype("category")

df["name"] = df["name"].str.strip()
df["ticket"] = df["ticket"].str.strip()

df.head()

,passenger_id,survived,passenger_class,name,sex,age,siblings_spouses,parents_children,ticket,fare,embarked_port,has_cabin
0,1,No,3,"Braund, Mr. Owen Harris",male,22,1,0,A/5 21171,7.2500,Southampton,0
1,2,Yes,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38,1,0,PC 17599,71.2833,Cherbourg,1
2,3,Yes,3,"Heikkinen, Miss. Laina",female,26,0,0,STON/O2. 3101282,7.9250,Southampton,0
3,4,Yes,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35,1,0,113803,53.1000,Southampton,1
4,5,No,3,"Allen, Mr. William Henry",male,35,0,0,373450,8.0500,Southampton,0


## Step 6: Final check
We confirm the data really is clean before saving it.

In [12]:
print("Missing values :", df.isnull().sum().sum())
print("Duplicate rows :", df.duplicated().sum())
print("Shape          :", df.shape)
print()
df.info()

Missing values : 0
Duplicate rows : 0
Shape          : (891, 12)

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   passenger_id      891 non-null    int64   
 1   survived          891 non-null    category
 2   passenger_class   891 non-null    category
 3   name              891 non-null    str     
 4   sex               891 non-null    category
 5   age               891 non-null    int64   
 6   siblings_spouses  891 non-null    int64   
 7   parents_children  891 non-null    int64   
 8   ticket            891 non-null    str     
 9   fare              891 non-null    float64 
 10  embarked_port     891 non-null    category
 11  has_cabin         891 non-null    int64   
dtypes: category(4), float64(1), int64(5), str(2)
memory usage: 59.4 KB


In [13]:
df.describe()

,passenger_id,age,siblings_spouses,parents_children,fare,has_cabin
count,891.000000,891.000000,891.000000,891.000000,891.000000,891.000000
mean,446.000000,29.131313,0.523008,0.381594,32.204208,0.228956
std,257.353842,13.289416,1.102743,0.806057,49.693429,0.420397
min,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,223.500000,22.000000,0.000000,0.000000,7.910400,0.000000
50%,446.000000,26.000000,0.000000,0.000000,14.454200,0.000000
75%,668.500000,36.000000,1.000000,0.000000,31.000000,0.000000
max,891.000000,80.000000,8.000000,6.000000,512.329200,1.000000


## Step 7: Save the clean dataset

In [14]:
df.to_csv("../data/titanic_clean.csv", index=False)
print("Saved: data/titanic_clean.csv")

Saved: data/titanic_clean.csv


## Summary: before vs after

| | Before | After |
|---|---|---|
| Rows | 891 | 891 |
| Columns | 12 | 12 (`Cabin` dropped, `has_cabin` added) |
| Missing values | 866 | 0 |
| Duplicate rows | 0 | 0 |
| Age type | decimal number | whole number |
| Category columns | stored as numbers or text | proper `category` type |
| Column names | mixed style, unclear (`SibSp`, `Parch`) | lowercase, readable |

**Decisions and why:**
- **Age:** median of each class and sex group, because the median is not distorted by very old passengers.
- **Embarked:** the most common port, because only 2 values were missing.
- **Cabin:** dropped because about 77% was missing, but a `has_cabin` flag was kept as a useful feature.